In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
from plotnine import *
from pathlib import Path
from tqdm import tqdm
import seaborn as sns
import numpy as np
from datetime import datetime
from enhance_ocod.analysis import create_summarised_stats, create_mean_difference_by_groups
from enhance_ocod.address_parsing import (
    process_addresses,
    expand_dataframe_numbers,
    create_unique_id
)

data_folder = Path('../data') 
figures_folder = Path('../figures/figures')
figures_folder.mkdir(parents=True, exist_ok=True)

OCOD_history_path = data_folder / 'ocod_history_processed' 
#
list_of_files = list(OCOD_history_path.iterdir())

active_class_var = 'class'


LAD_COLUMN_CODE = 'LAD22CD' # change this according to the shapefile you are using
LAD_COLUMN_NAME = "LAD22NM"

# What the issue with Guernsey?

In [ ]:
guernsey_res_df = []

for file in list_of_files:

    target_file = pd.read_parquet(file)

    target_file = target_file.loc[(target_file['country_incorporated']=='GUERNSEY') & 
    (target_file['class']=='residential')]

    target_file = target_file.groupby(['title_number', 'msoa11cd' ]).size().reset_index().rename(columns = {0:'counts'})
    target_file = target_file.sort_values('counts')
    target_file['file'] = Path(file).stem

    guernsey_res_df.append(target_file)

guernsey_res_df = pd.concat(guernsey_res_df, ignore_index = False)

guernsey_res_df['date'] = guernsey_res_df['file'].str.extract(r'(\d{4}_\d{2})$')[0]
guernsey_res_df['date'] = guernsey_res_df['date'].str.replace('_', '-') + '-01'
guernsey_res_df['date'] = pd.to_datetime(guernsey_res_df['date'])


In [ ]:
guernsey_large_only = guernsey_res_df.loc[guernsey_res_df['counts']>500]

In [ ]:
guernsey_large_only.sort_values('date')

In [ ]:
large_group_df = pd.read_parquet(list_of_files[50])
large_group_df = large_group_df.loc[large_group_df['title_number'] == 'AGL427518' ]

large_group_df['property_address'].iloc[0]

In [ ]:
large_group_df

In [ ]:
large_group_df.to_csv(str(data_folder)+'/guernsey_check.csv')

From the results we can see that title AGL427518 is being incorrectly parsed resulting in several thousand fake addresses. 

# Is this a parsing failure or a model failure?



In [ ]:
import pandas as pd
from enhance_ocod.inference import parse_addresses_basic

# Create example DataFrame with the two addresses
example_df = pd.DataFrame({
    'address': [
        "Apartments 201-209, 301-309, 401-409, 501-509, 601-609, 701-709, 801-809, 901-909, 1001-1009, 1101-1109, 1201-1209, 1301-1309, 1401-1409, 1501-1509, 1601-1609, 1701-1709, 1801-1809, 1901-1909, 2001-2009, 2201-2209, 2301-2309, 2401-2407, 2409, 2501-2509, 2601-2609, 2701-2709, 2801-2809, 2901-2909, 3001-3009, 3101-3109, 3201-3209, 3301-3309, 3401-3409, 3501-3509, 3601-3609, 3701-3709, 3801-3807, 3809, 3901-3909, 4001-4003, 4005-4009, 4101-4109, 4201-4209, 4301-4302 Arena Tower, 25 Crossharbour Plaza, London",
    ],
    'datapoint_id': ['addr_001']  # Optional unique identifiers
})

print("Example DataFrame:")
print(example_df)

# Default behaviour is to download the finetuned model from Hugginface model library.
results = parse_addresses_basic(example_df)
print(f"Parsed {results['summary']['successful_parses']} addresses")

In [ ]:
parsed_problem_address = process_addresses(results['results'])
parsed_problem_address['class'] = 'residential'
parsed_problem_address['number_filter'] = 'all'
parsed_problem_address

In [ ]:
problem_expanded = expand_dataframe_numbers(parsed_problem_address, class_var = 'class', print_every=1000000, min_count=1)

In [ ]:
problem_expanded

## The Guernsey Bump

There is a notable Jump in the number of properties registered in guernsey between January 2022 and Febuary 2022. The appearance came just before the introduction of the `Economic Crime (Transparency and Enforcement) Act 2022' however, due to the speed of the bill passing into law and that the data is only at monthly level it is not possible to see if there was likely to be a relationship

In [ ]:

def get_title_counts(file_path):
    """Reads a parquet file, filters for specific criteria, and counts titles."""
    df = pd.read_parquet(file_path)
    
    # Filter the data
    df_filtered = df.loc[
        (df['country_incorporated'] == 'GUERNSEY') & 
        (df['class'] == 'residential')
    ]
    
    # Group by the unique identifiers and get the counts
    df_counts = df_filtered.groupby(
        ['title_number', 'msoa11cd', 'region']
    ).size().reset_index(name='counts')
    
    return df_counts

file1_path = OCOD_history_path / 'OCOD_FULL_2022_01.parquet'
file2_path = OCOD_history_path / 'OCOD_FULL_2022_03.parquet'

first_df = get_title_counts(file1_path)
second_df = get_title_counts(file2_path)

merged_df = pd.merge(
    first_df,
    second_df,
    on=['title_number', 'msoa11cd', 'region'],
    how='outer',
    suffixes=('_Jan', '_Feb')
)

# --- Step 4: Calculate the difference ---
# The outer merge creates NaN for titles not present in one of the files.
# Fill these NaN values with 0 to correctly calculate the difference.
merged_df.fillna({'counts_Jan': 0, 'counts_Feb': 0}, inplace=True)

# Calculate the difference in counts
merged_df['count_difference'] = merged_df['counts_Feb'] - merged_df['counts_Jan']

# Convert count columns to integer type for cleanliness
merged_df['counts_Jan'] = merged_df['counts_Jan'].astype(int)
merged_df['counts_Feb'] = merged_df['counts_Feb'].astype(int)

# Display titles that only exist in one of the files
print("\nTitles with changes (newly added or removed):")
merged_df.sort_values('count_difference', ascending =False).head(20)

In [ ]:

second_df['counts'].sum()-first_df['counts'].sum()

In [ ]:
(second_df['counts'].sum()-first_df['counts'].sum())/second_df['counts'].sum()

In [ ]:
second_df.shape[0]-first_df['counts'].shape[0]

In [ ]:
(second_df.shape[0]-first_df['counts'].shape[0])/second_df.shape[0]

In [ ]:
df = pd.read_parquet(file2_path)
df.columns

In [ ]:

# --- Step 1: Modify the function to include 'country_incorporated' in the grouping ---
def get_title_counts_all_countries(file_path):
    """
    Reads a parquet file, filters for 'residential' class,
    and counts titles, grouped by country, title, MSOA, and region.
    """
    df = pd.read_parquet(file_path)
    
    # Filter only by 'class' now, as we want all countries
    df_filtered = df.loc[
        (df['class'] == 'residential')
    ]
    
    # Group by all relevant identifiers, including 'country_incorporated'
    df_counts = df_filtered.groupby(
        ['country_incorporated', 'title_number', 'msoa11cd', 'region'] 
    ).size().reset_index(name='counts')
    
    return df_counts

# --- Step 2: Define your file paths ---
file1_path = OCOD_history_path / 'OCOD_FULL_2022_01.parquet'
file2_path = OCOD_history_path / 'OCOD_FULL_2022_02.parquet'

# --- Step 3: Process each file to get counts for all countries ---
first_df = get_title_counts_all_countries(file1_path)
second_df = get_title_counts_all_countries(file2_path)

# --- Step 4: Merge the two dataframes, including 'country_incorporated' in the merge key ---
merged_df = pd.merge(
    first_df,
    second_df,
    on=['country_incorporated', 'title_number', 'msoa11cd', 'region'], # Added country_incorporated
    how='outer',
    suffixes=('_Jan', '_Feb')
)

# --- Step 5: Calculate the difference ---
# The outer merge creates NaN for entries not present in one of the files.
# Fill these NaN values with 0 to correctly calculate the difference.
merged_df.fillna({'counts_Jan': 0, 'counts_Feb': 0}, inplace=True)

# Calculate the difference in counts
merged_df['count_difference'] = merged_df['counts_Feb'] - merged_df['counts_Jan']

# Convert count columns to integer type for cleanliness
merged_df['counts_Jan'] = merged_df['counts_Jan'].astype(int)
merged_df['counts_Feb'] = merged_df['counts_Feb'].astype(int)


print("\nTop 10 differences (most increased counts):")
merged_df.sort_values('count_difference', ascending=False).head(10)


In [ ]:
print("\nTop 10 differences (most decreased counts):")
merged_df.sort_values('count_difference', ascending=True).head(10)

In [ ]:
# Assume 'my_archive.zip' contains 'data.csv'
zip_file_path = '/teamspace/studios/this_studio/enhance_ocod/data/ocod_history/OCOD_FULL_2022_02.zip'
csv_file_in_zip = 'OCOD_FULL_2022_02.csv' # pandas needs to know the specific file name within the zip


    # pandas can directly read a CSV from a zip file.
    # The 'compression' argument can be 'zip', 'gzip', 'bz2', 'xz'.
    # The 'archive_name' parameter specifies which file inside the zip to read.
df = pd.read_csv("/teamspace/studios/this_studio/enhance_ocod/data/Guernsey_bump/OCOD_FULL_2022_03.csv")

df = df.loc[df['Country Incorporated (1)']=='GUERNSEY']

df.columns

In [ ]:
merged_df.sort_values('count_difference', ascending =False).head(20)['title_number']

In [ ]:
df.loc[df['Title Number'].isin(['NGL942924'])]

In [ ]:
df.loc[df['Title Number'].isin(merged_df.sort_values('count_difference', ascending =False).head(20)['title_number'])]

# What is happening with Mauritiaus?

Between 2017 and 2018 there is a massive drop in multi-property registrations in mauritius losing approximately 8500 properties.

This was related to the closing of a tax loophole which would have meant the properties would be taxed at a minimum rate of 15% in the UK. To avoid this the properties were either sold, moved jurisdiction or transferred to a company based in the UK


In [ ]:
MAURITIUS_df = []

for file in [OCOD_history_path/'OCOD_FULL_2017_09.parquet', OCOD_history_path/'OCOD_FULL_2018_02.parquet']:

    target_file = pd.read_parquet(file)

    target_file = target_file.loc[(target_file['country_incorporated']=='MAURITIUS') & 
    (target_file['class']=='residential')]

    target_file = target_file.groupby(['title_number', 'msoa11cd', 'region' ]).size().reset_index().rename(columns = {0:'counts'})
    target_file = target_file.sort_values('counts')
    target_file['file'] = Path(file).stem

    MAURITIUS_df.append(target_file)

MAURITIUS_df = pd.concat(MAURITIUS_df, ignore_index = False)

MAURITIUS_df['date'] = MAURITIUS_df['file'].str.extract(r'(\d{4}_\d{2})$')[0]
MAURITIUS_df['date'] = MAURITIUS_df['date'].str.replace('_', '-') + '-01'
MAURITIUS_df['date'] = pd.to_datetime(MAURITIUS_df['date'])

In [ ]:
MAURITIUS_df.sort_values('counts',ascending = False)

In [ ]:
MAURITIUS_df.groupby('region').agg(region = ('region', 'count'))

In [ ]:
MAURITIUS_df.groupby('date').agg(
    properties=('counts', 'sum'),          
    titles=('date', lambda x: len(x))     
                                              
)

In [ ]:
MSOA_price_df = pd.read_parquet(data_folder / 'price_paid_msoa_averages/price_paid_2018_02.parquet')

Muaritius_with_price = MAURITIUS_df.merge(MSOA_price_df, on = 'msoa11cd')

Muaritius_with_price['total_value'] = Muaritius_with_price['counts'] * Muaritius_with_price['price_mean'] 

Muaritius_with_price['total_value'].sum()

In [ ]:
Muaritius_with_price

In [ ]:
print(1-871/9491)
print(1-319/1905)

Looking at the above results we can see that during this period there was a sudden decrease in the number of titles registered in mauritus.